In [ ]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

hemantthegood_flowers_data_path = kagglehub.dataset_download('hemantthegood/flowers-data')
print('Data source import complete.')


**In this notebook we will train model to classify flower in 14 categories and export it to onxx, to deploy.**

In [ ]:
import os
import numpy as np # linear algebra
from fastcore.all import *
from fastai.vision.all import *
import time, json
from pathlib import Path

In [ ]:
# Create Dataloader using DataBlock for more explicit control
path = Path(hemantthegood_flowers_data_path) # Ensure path is a Path object

dblock = DataBlock(
    blocks=(ImageBlock, CategoryBlock),
    get_items=get_image_files, # Function to find all image files
    splitter=RandomSplitter(valid_pct=0.2, seed=42), # Split data into training and validation
    get_y=parent_label, # Function to extract label from parent folder name (folder name is the class)
    item_tfms=Resize(512), # Resize images on CPU for initial processing
    batch_tfms=[
        *aug_transforms(
            size=224,           # Target input size for ConvNeXt, applied on GPU
            flip_vert=True,     # Complete orientation freedom for flowers
            max_rotate=180.0,   # Infinite 360-degree variations
            max_lighting=0.4,   # Robustness against strong sun/shadows
            max_zoom=1.3        # Force focus on internal petal details
        ),
        Normalize.from_stats(*imagenet_stats) # Normalize pixels based on ImageNet stats
    ]
)

# Create DataLoaders from the DataBlock and the data path
dls = dblock.dataloaders(path)
dls.show_batch()

**Train**

In [ ]:
learn = vision_learner(dls, convnext_tiny, metrics=[accuracy, error_rate])
learn.fine_tune(7)

In [ ]:
class_names = learn.dls.vocab
print(class_names)
import json
with open('class_names.json', 'w') as f:
    json.dump(list(class_names), f)

**Test**

In [ ]:
import torch
scripted_model = torch.jit.script(learn.model)
scripted_model.save("/content/flower-classification_convnext.pt")